In [1]:
# data manipulation
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from highlight_text import ax_text, fig_text

# animation
from PIL import Image, ImageSequence
import textwrap
import math
from matplotlib.animation import FuncAnimation

# OpenOA
from openoa.analysis.aep import MonteCarloAEP
from openoa.utils import filters, power_curve, plot

import project_ENGIE

ModuleNotFoundError: No module named 'highlight_text'

## Run the AEP analysis

In [ ]:
project = project_ENGIE.prepare('./data/la_haute_borne', use_cleansed=False)

In [ ]:
pa = project.MonteCarloAEP()

In [ ]:
pa.run(num_sim=5000)

In [ ]:
pa.results

## Animate the impacts of the number of simulations

In [ ]:
fig, [ax1, ax2, ax3] = plt.subplots(1, 3, figsize=(24, 8), sharey=True, squeeze=True, dpi=300, layout="tight")

xlim_aep=(6, 18)
xlim_availability=(0.008, 0.016)
xlim_curtail=(0.0004, 0.0008)

ylim_aep=(0, 800)
ylim_availability=(0, 800)
ylim_curtail=(0, 800)

xt1 = list(range(6, 19, 2))
xtl1 = xt1
xt2 = np.linspace(*xlim_availability, 9)
xtl2 = [f"{x:.1%}" for x in xt2]
xt3 = np.linspace(*xlim_curtail, 5)
xtl3 = [f"{x:.2%}" for x in xt3]

n_frames = 100
def update(frame):
    ax1.clear()
    ax2.clear()
    ax3.clear()
    n = int((1 + frame) / n_frames * pa.num_sim)
    subset_df = pa.results.loc[:n]
    ax1.hist(subset_df.aep_GWh, bins=40)
    ax2.hist(subset_df.avail_pct, bins=40)
    ax3.hist(subset_df.curt_pct, bins=40)

    ax1.set_xlim(*xlim_aep)
    ax2.set_xlim(*xlim_availability)
    ax3.set_xlim(*xlim_curtail)
    # ax1.set_ylim(*ylim_aep)
    # ax2.set_ylim(*ylim_availability)
    # ax3.set_ylim(*ylim_curtail)

    ax1.set_xticks(xt1)
    ax1.set_xticklabels(xtl1)
    ax2.set_xticks(xt2)
    ax2.set_xticklabels(xtl2)
    ax3.set_xticks(xt3)
    ax3.set_xticklabels(xtl3)

    ax1.set_xlabel("AEP (GWh/yr)")
    ax2.set_xlabel("Availability Loss (%)")
    ax3.set_xlabel("Curtailment Loss (%)")
    ax1.set_ylabel("Number of Samples")

    ax2.set_title(f"Number of Simulations: {n:,.0f}")

    vals = subset_df.aep_GWh.values
    u_vals = vals.mean()
    ax1.annotate(
        f"Mean = {u_vals:.1f}",
        (0.05, 0.9),
        xycoords="axes fraction",
        fontsize=12,
    )
    uncertainty = (vals.std() / u_vals) if u_vals != 0.0 else 0.0
    ax1.annotate(
        f"Uncertainty = {uncertainty:.1%}",
        (0.05, 0.85),
        xycoords="axes fraction",
        fontsize=12,
    )

    vals = subset_df.avail_pct.values
    u_vals = vals.mean()
    ax2.annotate(
        f"Mean = {u_vals:.1%}",
        (0.05, 0.9),
        xycoords="axes fraction",
        fontsize=12,
    )
    uncertainty = (vals.std() / u_vals) if u_vals != 0.0 else 0.0
    ax2.annotate(
        f"Uncertainty = {uncertainty:.1%}",
        (0.05, 0.85),
        xycoords="axes fraction",
        fontsize=12,
    )

    vals = subset_df.curt_pct.values
    u_vals = vals.mean()
    ax3.annotate(
        f"Mean = {u_vals:.3%}",
        (0.05, 0.9),
        xycoords="axes fraction",
        fontsize=12,
    )
    uncertainty = (vals.std() / u_vals) if u_vals != 0.0 else 0.0
    ax3.annotate(
        f"Uncertainty = {uncertainty:.1%}",
        (0.05, 0.85),
        xycoords="axes fraction",
        fontsize=12,
    )
    
    return ax1, ax2, ax3

ani = FuncAnimation(
    fig=fig,
    func=update,
    frames=n_frames,
    interval=1 # in ms
)
ani.save("aep_results.gif", writer='imagemagick', fps=3)

## Animate the stability of the parameters

In [ ]:
mc_reg = pd.DataFrame(data={
    'slope': pa._mc_slope.ravel(),
    'intercept': pa._mc_intercept, 
    'num_points': pa._mc_num_points, 
    'metered_energy_fraction': pa.mc_inputs.metered_energy_fraction, 
    'loss_fraction': pa.mc_inputs.loss_fraction,
    'num_years_windiness': pa.mc_inputs.num_years_windiness, 
    'loss_threshold': pa.mc_inputs.loss_threshold,
    'reanalysis_product': pa.mc_inputs.reanalysis_product
})
mc_reg

In [ ]:
fig, [[ax1, ax2, ax3, ax4], [ax5, ax6, ax7, ax8]] = plt.subplots(
    2, 4, figsize=(16, 8), sharey=False, squeeze=True, dpi=300, layout="tight"
)

xlim_s=(0.25, 0.5)
xlim_i=(-1.8, -0.6)
xlim_np=(23, 25)
xlim_mef=(0.98, 1.02)
xlim_lf=(0.8, 1.2)
xlim_nyw=(9, 21)
xlim_lt=(0.09, 0.21)
xlim_rp=(-1, 2)

xt1 = np.linspace(*xlim_s, 6)
xtl1 = [f"{x:.2f}" for x in xt1]
xt2 = np.linspace(*xlim_i, 7)
xtl2 = [f"{x:.1f}" for x in xt2]
xt3 = np.linspace(*xlim_np, 5)
xtl3 = [f"{x:.0f}" for x in xt3]
xt4 = np.linspace(*xlim_mef, 5)
xtl4 = [f"{x:.2f}" for x in xt4]
xt5 = np.linspace(*xlim_lf, 5)
xtl5 = [f"{x:.1f}" for x in xt5]
xt6 = np.linspace(*xlim_nyw, 5)
xtl6 = [f"{x:.0f}" for x in xt6]
xt7 = np.linspace(*xlim_lt, 7)
xtl7 = [f"{x:.2f}" for x in xt7]
# xt8 = np.linspace(*xlim_rp, 5)
# xtl8 = [f"{x:.2%}" for x in xt7]

n_frames = 50
def update(frame):
    ax1.clear()
    ax2.clear()
    ax3.clear()
    ax4.clear()
    ax5.clear()
    ax6.clear()
    ax7.clear()
    ax8.clear()
    
    n = int((1 + frame) / n_frames * pa.num_sim)
    subset_df = mc_reg.loc[:n]
    
    ax1.hist(subset_df.slope, bins=40)
    ax2.hist(subset_df.intercept, bins=40)
    ax3.hist(subset_df.num_points, bins=40)
    ax4.hist(subset_df.metered_energy_fraction, bins=40)
    ax5.hist(subset_df.loss_fraction, bins=40)
    ax6.hist(subset_df.num_years_windiness, bins=40)
    ax7.hist(subset_df.loss_threshold, bins=40)
    ax8.hist(subset_df.reanalysis_product, bins=40)
    
    ax1.set_xlim(*xlim_s)
    ax2.set_xlim(*xlim_i)
    ax3.set_xlim(*xlim_np)
    ax4.set_xlim(*xlim_mef)
    ax5.set_xlim(*xlim_lf)
    ax6.set_xlim(*xlim_nyw)
    ax7.set_xlim(*xlim_lt)
    ax8.set_xlim(*xlim_rp)
    
    ax1.set_xticks(xt1)
    ax1.set_xticklabels(xtl1)
    ax2.set_xticks(xt2)
    ax2.set_xticklabels(xtl2)
    ax3.set_xticks(xt3)
    ax3.set_xticklabels(xtl3)
    ax4.set_xticks(xt4)
    ax4.set_xticklabels(xtl4)
    ax5.set_xticks(xt5)
    ax5.set_xticklabels(xtl5)
    ax6.set_xticks(xt6)
    ax6.set_xticklabels(xtl6)
    ax7.set_xticks(xt7)
    ax7.set_xticklabels(xtl7)

    ax2.set_title(f"Number of Simulations: {n:,.0f}")
    ax1.set_xlabel("Slope")
    ax2.set_xlabel("Intercept")
    ax3.set_xlabel("Number of Points")
    ax4.set_xlabel("Metered Energy Fraction")
    ax5.set_xlabel("Loss Fraction")
    ax6.set_xlabel("Number of Years in Windiness Correction")
    ax7.set_xlabel("Loss Threshold")
    ax8.set_xlabel("Reanalysis Product")
    
    ax1.set_ylabel("Number of Samples")
    ax5.set_ylabel("Number of Samples")
   
    return ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8

ani = FuncAnimation(
    fig=fig,
    func=update,
    frames=n_frames,
    interval=1 # in ms
)
ani.save("linear_model_params.gif", writer='imagemagick', fps=3)

## Animate the turbine power curve filtering

### Create the data

In [ ]:
# Get the turbine data
turb_list = project.turbine_ids
df1 = project.scada.loc[project.scada.index.get_level_values("asset_id") == turb_list[0]]
df2 = project.scada.loc[project.scada.index.get_level_values("asset_id") == turb_list[1]]
df3 = project.scada.loc[project.scada.index.get_level_values("asset_id") == turb_list[2]]
df4 = project.scada.loc[project.scada.index.get_level_values("asset_id") == turb_list[3]]
print(df1.shape, df2.shape, df3.shape, df4.shape)

# Remove bad data and out of range (if any)
nan1 = df1.WMET_HorWdSpd.isna() | df1.WTUR_W.isna()
ws1 = df1.loc[~nan1, "WMET_HorWdSpd"].copy()
pw1 = df1.loc[~nan1, "WTUR_W"].copy()

nan2 = df2.WMET_HorWdSpd.isna() | df2.WTUR_W.isna()
ws2 = df2.loc[~nan2, "WMET_HorWdSpd"].copy()
pw2 = df2.loc[~nan2, "WTUR_W"].copy()

nan3 = df3.WMET_HorWdSpd.isna() | df3.WTUR_W.isna()
ws3 = df3.loc[~nan3, "WMET_HorWdSpd"].copy()
pw3 = df3.loc[~nan3, "WTUR_W"].copy()

nan4 = df4.WMET_HorWdSpd.isna() | df4.WTUR_W.isna()
ws4 = df4.loc[~nan4, "WMET_HorWdSpd"].copy()
pw4 = df4.loc[~nan4, "WTUR_W"].copy()

#out_of_range = filters.range_flag(windspeed, lower=0, upper=70)
#windspeed[out_of_range].head()

In [ ]:
# Out of window
out_of_window1 = filters.window_range_flag(ws1, 5., 40, pw1, 20., 2100.)
out_of_window2 = filters.window_range_flag(ws2, 5., 40, pw2, 20., 2100.)
out_of_window3 = filters.window_range_flag(ws3, 5., 40, pw3, 20., 2100.)
out_of_window4 = filters.window_range_flag(ws4, 5., 40, pw4, 20., 2100.)

# Remove out of window
ws1_f1 = ws1[~out_of_window1]
pw1_f1 = pw1[~out_of_window1]
ws2_f1 = ws2[~out_of_window2]
pw2_f1 = pw2[~out_of_window2]
ws3_f1 = ws3[~out_of_window3]
pw3_f1 = pw3[~out_of_window3]
ws4_f1 = ws4[~out_of_window4]
pw4_f1 = pw4[~out_of_window4]

# Bin outliers
max_bin1 = 0.90 * pw1_f1.max()
bin_outliers1 = filters.bin_filter(pw1_f1, ws1_f1, 100, 1.5, "median", 20., max_bin1, "scalar", "all")
max_bin2 = 0.90 * pw2_f1.max()
bin_outliers2 = filters.bin_filter(pw2_f1, ws2_f1, 100, 1.5, "median", 20., max_bin2, "scalar", "all")
max_bin3 = 0.90 * pw3_f1.max()
bin_outliers3 = filters.bin_filter(pw3_f1, ws3_f1, 100, 1.5, "median", 20., max_bin3, "scalar", "all")
max_bin4 = 0.90 * pw4_f1.max()
bin_outliers4 = filters.bin_filter(pw4_f1, ws4_f1, 100, 1.5, "median", 20., max_bin4, "scalar", "all")

# Remove bin outliers
ws1_f2 = ws1_f1[~bin_outliers1]
pw1_f2 = pw1_f1[~bin_outliers1]
ws2_f2 = ws2_f1[~bin_outliers2]
pw2_f2 = pw2_f1[~bin_outliers2]
ws3_f2 = ws3_f1[~bin_outliers3]
pw3_f2 = pw3_f1[~bin_outliers3]
ws4_f2 = ws4_f1[~bin_outliers4]
pw4_f2 = pw4_f1[~bin_outliers4]

# Unresponsive readings
frozen1 = filters.unresponsive_flag(ws1_f2, 3)
frozen2 = filters.unresponsive_flag(ws2_f2, 3)
frozen3 = filters.unresponsive_flag(ws3_f2, 3)
frozen4 = filters.unresponsive_flag(ws4_f2, 3)

# Remove unresponsive sensors
ws1_final = ws1_f2[~frozen1]
pw1_final = pw1_f2[~frozen1]
ws2_final = ws2_f2[~frozen2]
pw2_final = pw2_f2[~frozen2]
ws3_final = ws3_f2[~frozen3]
pw3_final = pw3_f2[~frozen3]
ws4_final = ws4_f2[~frozen4]
pw4_final = pw4_f2[~frozen4]

# Apply the power curves
iec_curve1 = power_curve.IEC(ws1_final, pw1_final)
l5p_curve1 = power_curve.logistic_5_parametric(ws1_final, pw1_final)
spline_curve1 = power_curve.gam(ws1_final, pw1_final, n_splines=20)
iec_curve2 = power_curve.IEC(ws2_final, pw2_final)
l5p_curve2 = power_curve.logistic_5_parametric(ws2_final, pw2_final)
spline_curve2 = power_curve.gam(ws2_final, pw2_final, n_splines=20)
iec_curve3 = power_curve.IEC(ws3_final, pw3_final)
l5p_curve3 = power_curve.logistic_5_parametric(ws3_final, pw3_final)
spline_curve3 = power_curve.gam(ws3_final, pw3_final, n_splines=20)
iec_curve4 = power_curve.IEC(ws4_final, pw4_final)
l5p_curve4 = power_curve.logistic_5_parametric(ws4_final, pw4_final)
spline_curve4 = power_curve.gam(ws4_final, pw4_final, n_splines=20)

### Plot the power curve filtering progression

In [ ]:
def plot1():
    ax1.scatter(ws1, pw1, alpha=0.4, s=10)
    ax2.scatter(ws2, pw2, alpha=0.4, s=10)
    ax3.scatter(ws3, pw3, label="Acceptable Power Curve Points", alpha=0.4, s=10)
    ax4.scatter(ws4, pw4, alpha=0.4, s=10)
    return

def plot2():
    plot1()
    ax1.scatter(ws1[out_of_window1], pw1[out_of_window1], alpha=0.4, s=10)
    ax2.scatter(ws2[out_of_window2], pw2[out_of_window2], alpha=0.4, s=10)
    ax3.scatter(ws3[out_of_window3], pw3[out_of_window3], label="Window Filter", alpha=0.4, s=10)
    ax4.scatter(ws4[out_of_window4], pw4[out_of_window4], alpha=0.4, s=10)
    return

def plot3():
    plot2()
    ax1.scatter(ws1_f1[bin_outliers1], pw1_f1[bin_outliers1], alpha=0.4, s=10)
    ax2.scatter(ws2_f1[bin_outliers2], pw2_f1[bin_outliers2], alpha=0.4, s=10)
    ax3.scatter(ws3_f1[bin_outliers3], pw3_f1[bin_outliers3], label="Bin Filter", alpha=0.4, s=10)
    ax4.scatter(ws4_f1[bin_outliers4], pw4_f1[bin_outliers4], alpha=0.4, s=10)
    return

def plot4():
    plot3()
    ax1.scatter(ws1_f2[frozen1], pw1_f2[frozen1], alpha=0.4, s=10)
    ax2.scatter(ws2_f2[frozen2], pw2_f2[frozen2], alpha=0.4, s=10)
    ax3.scatter(ws3_f2[frozen3], pw3_f2[frozen3], label="Unresponsive Filter", alpha=0.4, s=10)
    ax4.scatter(ws4_f2[frozen4], pw4_f2[frozen4], alpha=0.4, s=10)
    return

def plot5():
    plot4()
    ax1.plot(x, iec_curve1(x), color="tab:orange", linewidth = 2)
    ax2.plot(x, iec_curve2(x), color="tab:orange", linewidth = 2)
    ax3.plot(x, iec_curve3(x), color="tab:orange", linewidth = 2)
    ax4.plot(x, iec_curve4(x), color="tab:orange", label = "IEC", linewidth = 2)
    return
    return

def plot6():
    plot5()
    ax1.plot(x, spline_curve1(x), color="tab:green", linewidth = 2)
    ax2.plot(x, spline_curve2(x), color="tab:green", linewidth = 2)
    ax3.plot(x, spline_curve3(x), color="tab:green", linewidth = 2)
    ax4.plot(x, spline_curve4(x), color="tab:green", label = "Spline", linewidth = 2)
    return
    return

def plot7():
    plot6()
    ax1.plot(x, l5p_curve1(x), color="tab:red", linewidth = 2)
    ax2.plot(x, l5p_curve2(x), color="tab:red", linewidth = 2)
    ax3.plot(x, l5p_curve3(x), color="tab:red", linewidth = 2)
    ax4.plot(x, l5p_curve4(x), color="tab:red", label = "L5P", linewidth = 2)
    return

In [ ]:
fig, [[ax1, ax2], [ax3, ax4]] = plt.subplots(
    2, 2, figsize=(16, 11), squeeze=True, dpi=300, layout="tight"
)
axes = [ax1, ax2, ax3, ax4]

# Setup the frame
xlim=(-1, 20)
ylim=(-100, 2100)

def update(frame):
    ax1.clear()
    ax2.clear()
    ax3.clear()
    ax4.clear()

    for i, ax in enumerate(axes):
        ax.annotate(
            f"Turbine ID: {turb_list[i]}",
            (0.05, 0.85),
            xycoords="axes fraction",
            fontsize=12,
        )
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
    
    ax1.set_ylabel("Power (kW)")
    ax3.set_ylabel("Power (kW)")
    ax3.set_xlabel("Wind Speed (m/s)")
    ax4.set_xlabel("Wind Speed (m/s)")
    
    if frame == 0:
        plot1()
        ax3.legend(loc="lower left", bbox_to_anchor=(0, -0.3))
        # leg = fig.legend(loc="lower center", bbox_to_anchor=(0.25, -0.05))
    elif frame == 1:
        plot2()
        ax3.legend(loc="lower left", bbox_to_anchor=(0, -0.3), ncol=2)
        # leg = fig.legend(loc="lower center", bbox_to_anchor=(0.25, -0.05), ncol=2)
    elif frame == 2:
        plot3()
        ax3.legend(loc="lower left", bbox_to_anchor=(0, -0.3), ncol=2)
        # leg = fig.legend(loc="lower center", bbox_to_anchor=(0.25, -0.05), ncol=3)
    elif frame == 3:
        plot4()
        ax3.legend(loc="lower left", bbox_to_anchor=(0, -0.3), ncol=2)
        # leg = fig.legend(loc="lower center", bbox_to_anchor=(0.25, -0.05), ncol=4)
    elif frame == 4:
        plot5()
        ax3.legend(loc="lower left", bbox_to_anchor=(0, -0.3), ncol=2)
        ax4.legend(loc="lower left", bbox_to_anchor=(0, -0.25), ncol=1)
        # leg = fig.legend(loc="lower center", bbox_to_anchor=(0.25, -0.1), ncol=4)
    elif frame == 5:
        plot6()
        ax3.legend(loc="lower left", bbox_to_anchor=(0, -0.3), ncol=2)
        ax4.legend(loc="lower left", bbox_to_anchor=(0, -0.25), ncol=2)
        # leg = fig.legend(loc="lower center", bbox_to_anchor=(0.25, -0.1), ncol=4)
    elif frame == 6:
        plot7()
        ax3.legend(loc="lower left", bbox_to_anchor=(0, -0.3), ncol=2)
        ax4.legend(loc="lower left", bbox_to_anchor=(0, -0.25), ncol=3)
        # leg = fig.legend(loc="lower center", bbox_to_anchor=(0.25, -0.1), ncol=4)
    
    return ax1, ax2, ax3, ax4

ani = FuncAnimation(
    fig=fig,
    func=update,
    frames=7,
    interval=1
)
ani.save("power_curve_filtering.gif", writer='imagemagick', fps=0.25)

In [ ]:
ev = pd.read_csv(
    "/Users/rhammond/Downloads/2024-06-13_14-09-53_COREWIND_Morro_Bay_In_Situ_power_potential/2024-06-13_14-09-53_COREWIND_Morro_Bay_In_Situ_events.csv",
    sep="|",
    parse_dates=["datetime", "env_datetime"],
).set_index("datetime")

In [ ]:
op = pd.read_csv(
    "/Users/rhammond/Downloads/2024-06-13_14-09-53_COREWIND_Morro_Bay_In_Situ_power_potential/2024-06-13_14-09-53_COREWIND_Morro_Bay_In_Situ_operations.csv",
    sep="|",
    parse_dates=True,
).set_index("datetime")

In [ ]:
production = pd.read_csv(
    "/Users/rhammond/Downloads/2024-06-13_14-09-53_COREWIND_Morro_Bay_In_Situ_power_potential/2024-06-13_14-09-53_COREWIND_Morro_Bay_In_Situ_power_production.csv",
    sep="|",
    parse_dates=True,
)

In [ ]:
production.loc[
    (production.windfarm <= (1140000 / 2))
    & (production.env_datetime >= "2009-11-1")
]

In [ ]:
op.loc[op.env_datetime >= "2009-11-10"].head(24)

In [ ]:
ev.loc[ev.env_datetime >= "2009-11-10"].head(20)

In [ ]:
timing = ev.loc[ev.action.isin(("repair request", "maintenance request", "repair complete", "maintenance complete"))]
timing = timing[["action", "duration"]].groupby([timing.env_datetime.dt.year, "action"]).count().unstack().fillna(0)
timing.columns = timing.columns.droplevel(0)
timing["repairs remaining"] = timing["repair request"].cumsum() - timing["repair complete"].cumsum()
timing

In [ ]:
pd.options.display.max_rows=200